In [0]:
# Create or refresh the monitoring view
spark.sql("""
CREATE OR REPLACE VIEW
fraud_detection.gold.fraud_pipeline_monitoring AS

SELECT
    CURRENT_TIMESTAMP() AS monitoring_timestamp,

    (SELECT COUNT(*)
     FROM fraud_detection.bronze.realtime_transactions)
        AS bronze_transactions,

    (SELECT COUNT(*)
     FROM fraud_detection.silver.realtime_transactions)
        AS valid_transactions,

    (SELECT COUNT(*)
     FROM fraud_detection.quarantine.realtime_transactions)
        AS quarantined_transactions,

    (SELECT COUNT(*)
     FROM fraud_detection.gold.realtime_scored_transactions)
        AS scored_transactions,

    (SELECT COUNT(*)
     FROM fraud_detection.gold.realtime_fraud_alerts)
        AS fraud_alerts,

    (SELECT COUNT(*)
     FROM fraud_detection.gold.fraud_cases
     WHERE case_status = 'open')
        AS open_cases,

    (SELECT COUNT(*)
     FROM fraud_detection.gold.fraud_cases
     WHERE case_status LIKE '%confirmed_fraud%')
        AS confirmed_fraud_cases,

    (SELECT COALESCE(SUM(disputed_amount), 0)
     FROM fraud_detection.gold.fraud_disputes)
        AS total_disputed_amount
""")

In [0]:
reconciliation = spark.sql("""
SELECT
    bronze_transactions,
    valid_transactions,
    quarantined_transactions,
    scored_transactions,
    fraud_alerts,
    valid_transactions + quarantined_transactions
        AS accounted_transactions
FROM fraud_detection.gold.fraud_pipeline_monitoring
""").first()

bronze_count = reconciliation["bronze_transactions"]
valid_count = reconciliation["valid_transactions"]
quarantine_count = reconciliation["quarantined_transactions"]
scored_count = reconciliation["scored_transactions"]
alert_count = reconciliation["fraud_alerts"]

print("Bronze:", bronze_count)
print("Silver valid:", valid_count)
print("Quarantine:", quarantine_count)
print("Gold scored:", scored_count)
print("Fraud alerts:", alert_count)

In [0]:
if bronze_count != valid_count + quarantine_count:
    raise Exception(
        "Ingestion reconciliation failed: "
        "Bronze does not equal Silver plus Quarantine."
    )

if valid_count != scored_count:
    raise Exception(
        "Scoring reconciliation failed: "
        "not every valid transaction was scored."
    )

print("Ingestion reconciliation: PASSED")
print("Scoring reconciliation: PASSED")
print("Pipeline monitoring completed successfully")